[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_85_Dense_and_Hybrid_Retrieval.ipynb)

# Lesson 85 — Dense & Hybrid Retrieval
### Phase 10 · RAG at Production Scale · Lesson 3 of ~7

> **The one idea:** In L83 and L84 we held the *retriever* constant (a lexical TF-IDF index) and
> studied the pipeline and the chunking around it. Today the retriever itself is the variable. A
> lexical retriever matches **words**; it is blind to **meaning**. Ask it *"how do I reset my
> password"* and it scores the *credential-recovery* doc at **0.000** — because that doc never uses
> the words "reset" or "password". This is the **semantic gap (Failure Mode #2)**. We close it with
> **dense embeddings**, keep lexical's exact-match precision with **BM25**, and get the best of both
> with a **hybrid** retriever fused by **Reciprocal Rank Fusion (RRF)**.

**Where this sits.** L84 proved chunking sets an *unrecoverable ceiling*: the retriever can only rank
chunks that exist. Today we assume good chunks (we reuse L84's winning *recursive* splitter, held
constant) and ask the next question: **given good chunks, how do we rank the RIGHT one first?**


## Phase 10 roadmap — the RAG track

| # | Lesson | Failure mode it attacks | Status |
|---|--------|-------------------------|--------|
| L83 | RAG production baseline — where naive RAG breaks | pipeline / retrieval framing | ✅ |
| L84 | Chunking strategies — the ceiling is set at index time | FM4: chunk size & boundaries | ✅ |
| **L85** | **Dense & hybrid retrieval — embeddings + BM25 + RRF** | **FM2: the semantic gap** | **← you are here** |
| L86 | Cross-encoder reranking — reorder the top-N precisely | FM3: right chunk retrieved but ranked #7 | ⏭ next |
| L87 | Grounding & citations — force answers from evidence | FM5: hallucination / no provenance | ⏳ |
| L88 | RAG evaluation — faithfulness, context precision/recall | FM6: "is it actually better?" | ⏳ |
| L89 | Phase-10 capstone — a production RAG service | integrate everything | ⏳ |

**Design (mirrors L84).** L84 fixed the retriever and varied the *chunker*. Today we fix the chunker
(L84's `recursive`, which scored a perfect `answer_in_top1`) and vary the *retriever*. One variable at
a time — that is how you attribute a result to a cause instead of guessing.


## Setup

One `pip` cell. **No API key is needed today** — retrieval is pure math (embeddings + BM25 + fusion),
no LLM calls.

The dense retriever uses **`sentence-transformers`** with the small, fast **`all-MiniLM-L6-v2`** model
(~80 MB, downloads once, runs on CPU in Colab in seconds). If for any reason the model can't be
downloaded (e.g. an offline sandbox), the notebook automatically falls back to a deterministic local
stand-in so **every cell still runs** — but you only see the *real* semantic win with the real model,
so run this in Colab.

In [ ]:
!pip install sentence-transformers rank_bm25 scikit-learn numpy -q

import re, numpy as np
np.set_printoptions(precision=3, suppress=True)
print("imports ready")

## §0 — Recap: the retriever we have been carrying since L83

Since L83 our retriever has been **TF-IDF + cosine similarity**. It represents every chunk (and the
query) as a sparse vector of word weights and ranks by cosine. It is fast, needs no model, and is a
perfectly good *lexical* matcher — if the query and the answer share words.

The problem is that **users do not use your documents' vocabulary.** Your docs say *"credential
recovery flow … set a new secret"*; the user types *"reset my password"*. Zero important words in
common → cosine ≈ 0 → the right document is **invisible**, no matter how well you chunked it. That is
**Failure Mode #2, the semantic gap**, and it is the single biggest reason "our RAG can't find
obvious things" tickets exist.

Two fixes, and today we build both and combine them:

- **Dense retrieval** — embed text with a neural model so *meaning* (not words) drives similarity.
  "reset my password" and "credential recovery" land near each other in vector space. Fixes FM2.
- **BM25** — a much stronger *lexical* ranker than raw TF-IDF (it saturates term frequency and
  normalizes by document length). Still lexical, so it does NOT fix FM2 — but it is unbeatable at
  exact tokens like `HTTP 429`, error codes, IDs, and rare numbers that dense models blur.

Then **hybrid (RRF)** fuses the two ranked lists: dense supplies *recall on meaning*, BM25 supplies
*precision on exact tokens*.

## §1 — The corpus and the eval (identical to L83/L84)

Same eight Nimbus support docs. Same eight labelled questions (`EVAL`) — each names the `gold` doc and
the full `expect` phrase. **New today:** a second eval set, `PARAPHRASE`, where the question shares
almost no words with the answer doc. This is the semantic gap, made measurable.

In [ ]:
DOCS = {
 "refunds": ("Nimbus Refund Policy. Customers on the monthly plan may request a full refund "
   "within 14 days of any charge. Annual plans are refundable on a prorated basis for the "
   "remaining unused months. Refunds are issued to the original payment method and take 5 to "
   "10 business days to appear. One-time setup fees are non-refundable."),
 "ratelimits": ("Nimbus API Rate Limits. The Free tier allows 60 requests per minute. The Pro "
   "tier allows 600 requests per minute. The Enterprise tier allows 6000 requests per minute. "
   "Exceeding your limit returns HTTP 429. Each 429 response includes a Retry-After header "
   "telling you how many seconds to wait before retrying."),
 "sso": ("Nimbus Single Sign-On. SSO is available on the Enterprise plan only. We support SAML "
   "2.0 and OIDC. To configure SAML, an administrator uploads the identity provider metadata XML "
   "in the Security settings page. Just-in-time user provisioning is enabled by default so new "
   "users are created on first login."),
 "retention": ("Nimbus Data Retention. Application logs are retained for 30 days. Deleted "
   "projects are held in a recoverable trash state for 90 days before permanent deletion. "
   "Customers on the Enterprise plan can configure a custom retention window of up to 7 years "
   "for compliance."),
 "security": ("Nimbus Security. All customer data is encrypted at rest using AES-256 and in "
   "transit using TLS 1.3. Nimbus is SOC 2 Type II certified. Access to production systems "
   "requires hardware security keys. We run third-party penetration tests twice per year."),
 "credentials": ("Nimbus Account Access. If you are locked out, use the credential recovery flow "
   "on the sign-in page: enter your email and we send a one-time link that lets you set a new "
   "secret. Links expire after 30 minutes. Enabling two-factor authentication is strongly "
   "recommended for all accounts."),
 "pricing": ("Nimbus Pricing. The Free tier costs nothing and includes one project. The Pro tier "
   "costs 49 dollars per month and includes ten projects. The Enterprise tier is custom-priced "
   "and includes unlimited projects, SSO, and a dedicated support manager."),
 "support": ("Nimbus Support SLAs. Free tier support is community-only. Pro tier guarantees a "
   "first response within one business day. Enterprise tier guarantees a first response within "
   "one hour for urgent issues, twenty-four hours a day, seven days a week."),
}

# The original eval: the question already shares words with the answer (lexical-friendly).
EVAL = [
 {"q":"how many days to get a refund on a monthly plan","gold":"refunds","expect":"14 days"},
 {"q":"what is the Pro tier rate limit","gold":"ratelimits","expect":"600 requests per minute"},
 {"q":"which plans include single sign-on","gold":"sso","expect":"Enterprise"},
 {"q":"how long are deleted projects recoverable","gold":"retention","expect":"90 days"},
 {"q":"what encryption is used for data at rest","gold":"security","expect":"AES-256"},
 {"q":"how much does the Pro plan cost per month","gold":"pricing","expect":"49 dollars"},
 {"q":"how fast does Enterprise support respond to urgent issues","gold":"support","expect":"one hour"},
 {"q":"what is the maximum custom retention window for compliance","gold":"retention","expect":"7 years"},
]

# The semantic-gap eval: everyday phrasing that barely overlaps the docs' vocabulary.
PARAPHRASE = [
 {"q":"I forgot how to log in, how do I reset my password","gold":"credentials"},
 {"q":"my app keeps getting blocked after too many calls","gold":"ratelimits"},
 {"q":"can I get my money back","gold":"refunds"},
 {"q":"how do you keep my information safe from hackers","gold":"security"},
 {"q":"log in with my company account","gold":"sso"},
 {"q":"when will you delete my stuff for good","gold":"retention"},
]

print(f"{len(DOCS)} documents, {len(EVAL)} lexical-friendly questions, {len(PARAPHRASE)} paraphrase questions.")

## §2 — Chunk once, with L84's winner, then hold it constant

L84 showed the **recursive** splitter (LangChain's default: break on the biggest natural separator
that fits, recurse into smaller ones) scored a perfect `answer_in_top1`. We reuse it verbatim and
freeze it. From here on, the *only* thing that changes is the retriever.

In [ ]:
SEPARATORS = ["\n\n", "\n", ". ", " ", ""]

def _recurse(text, size, seps):
    if len(text) <= size:
        return [text]
    if not seps or seps[0] == "":                       # no separators left: hard char cut
        return [text[i:i+size] for i in range(0, len(text), size)]
    sep = seps[0]
    parts = text.split(sep)
    chunks, cur = [], ""
    for p in parts:
        cand = p if cur == "" else cur + sep + p
        if len(cand) <= size:
            cur = cand                                  # still fits: keep packing
        else:
            if cur:
                chunks.append(cur)
            if len(p) > size:
                chunks.extend(_recurse(p, size, seps[1:]))  # single part too big: recurse
                cur = ""
            else:
                cur = p
    if cur:
        chunks.append(cur)
    return chunks

def chunk_recursive(doc_id, text, size=180):
    out = []
    for piece in _recurse(text, size, SEPARATORS):
        piece = piece.strip()
        if piece:
            out.append({"id": f"{doc_id}#{len(out)}", "doc": doc_id, "text": piece})
    return out

CHUNKS = []
for d, t in DOCS.items():
    CHUNKS.extend(chunk_recursive(d, t, size=180))

TEXTS = [c["text"] for c in CHUNKS]
print(f"{len(CHUNKS)} chunks (frozen for the rest of the lesson).")
for c in CHUNKS[:3]:
    print(" ", c["id"], "->", c["text"][:70], "...")

## §3 — The lexical retrievers, and where they go blind

We build two lexical retrievers and watch **both** fail the same way on a paraphrase.

- **TF-IDF + cosine** — our L83/L84 retriever.
- **BM25** — the standard lexical ranker in real search engines. A *better algorithm* than TF-IDF,
  but still purely lexical.

The lesson of this cell: **a better lexical algorithm does not fix a semantic problem.** BM25 beats
TF-IDF on word-overlap queries yet is still ~0 when the words simply are not there.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi

def tok(s):
    return re.findall(r"[a-z0-9]+", s.lower())

class TfidfRetriever:
    name = "tfidf"
    def __init__(self, chunks):
        self.chunks = chunks
        self.vec = TfidfVectorizer(stop_words="english")
        self.M = self.vec.fit_transform([c["text"] for c in chunks])
    def scores(self, q):
        return cosine_similarity(self.vec.transform([q]), self.M)[0]
    def search(self, q, k=5):
        s = self.scores(q); order = np.argsort(-s)[:k]
        return [(self.chunks[i], float(s[i])) for i in order]

class BM25Retriever:
    name = "bm25"
    def __init__(self, chunks):
        self.chunks = chunks
        self.bm25 = BM25Okapi([tok(c["text"]) for c in chunks])
    def scores(self, q):
        return np.array(self.bm25.get_scores(tok(q)))
    def search(self, q, k=5):
        s = self.scores(q); order = np.argsort(-s)[:k]
        return [(self.chunks[i], float(s[i])) for i in order]

tfidf = TfidfRetriever(CHUNKS)
bm25  = BM25Retriever(CHUNKS)

gap_q = "I forgot how to log in, how do I reset my password"   # gold = credentials
print("QUERY:", gap_q, "\n")
for r in (tfidf, bm25):
    top, sc = r.search(gap_q, 1)[0]
    print(f"  {r.name:6} top-1 -> {top['id']:16} score={sc:.3f}   {'HIT' if top['doc']=='credentials' else 'MISS'}")
print("\nBoth lexical retrievers MISS: 'reset / password / forgot' appear in NO chunk of the")
print("credentials doc, which says 'credential recovery ... set a new secret'. Words != meaning.")

## §4 — Dense retrieval: ranking by meaning

A **sentence embedding** model maps text to a vector such that *semantically similar* text lands
nearby — even with zero shared words. We embed every chunk once (offline, at index time), embed the
query at search time, and rank by cosine.

`all-MiniLM-L6-v2` outputs 384-dimensional vectors. We L2-normalize so cosine similarity is just a
dot product. Watch the same paraphrase query that both lexical retrievers missed now land on the
`credentials` doc.

In [ ]:
# Try the real model; fall back to a deterministic local stand-in if it can't be downloaded
# (e.g. an offline sandbox). The fallback lets every cell run, but only the real model shows the
# true semantic win -> run this notebook in Colab for the real result.
MODEL_LOADED = False
try:
    from sentence_transformers import SentenceTransformer
    _model = SentenceTransformer("all-MiniLM-L6-v2")
    def embed(texts):
        return np.asarray(_model.encode(list(texts), normalize_embeddings=True), dtype="float32")
    MODEL_LOADED = True
    print("Dense backend: REAL sentence-transformers all-MiniLM-L6-v2 (semantic).")
except Exception as e:
    from sklearn.feature_extraction.text import TfidfVectorizer as _TV
    _fallback = _TV(analyzer="char_wb", ngram_range=(3, 5))
    _fallback.fit(TEXTS)
    def embed(texts):
        X = _fallback.transform(list(texts)).toarray().astype("float32")
        n = np.linalg.norm(X, axis=1, keepdims=True); n[n == 0] = 1.0
        return X / n
    print("Dense backend: FALLBACK char-ngram stand-in (NOT semantic; model unavailable:",
          type(e).__name__ + "). Run in Colab for the real result.")

class DenseRetriever:
    name = "dense"
    def __init__(self, chunks):
        self.chunks = chunks
        self.E = embed([c["text"] for c in chunks])       # (n_chunks, dim), embedded ONCE
    def scores(self, q):
        return self.E @ embed([q])[0]                     # cosine via dot on normalized vectors
    def search(self, q, k=5):
        s = self.scores(q); order = np.argsort(-s)[:k]
        return [(self.chunks[i], float(s[i])) for i in order]

dense = DenseRetriever(CHUNKS)

print("\nQUERY:", gap_q)
top, sc = dense.search(gap_q, 1)[0]
print(f"  dense  top-1 -> {top['id']:16} score={sc:.3f}   {'HIT' if top['doc']=='credentials' else 'MISS'}")
print("  text:", top["text"][:90], "...")
if MODEL_LOADED:
    print("\nThe semantic gap is closed: no shared words, yet the meaning matches.")

## §5 — Dense is not a free lunch: the exact-token weakness

Dense models are trained on *meaning*, so they can **blur rare, precise tokens** — error codes,
version strings, IDs, exact numbers. Ask for `HTTP 429` and a dense model may drift toward
semantically "related" text (retries, limits) while BM25 locks straight onto the literal token.

This is the honest mirror of §3: just as a better *lexical* algorithm doesn't fix meaning, a better
*semantic* model doesn't guarantee exact-token precision. Neither retriever dominates — which is
exactly why we fuse them.

In [ ]:
exact_q = "HTTP 429"   # a literal token; gold = ratelimits
print("QUERY:", exact_q, "\n")
for r in (bm25, dense):
    top, sc = r.search(exact_q, 1)[0]
    tag = "HIT" if top["doc"] == "ratelimits" else "MISS"
    print(f"  {r.name:6} top-1 -> {top['id']:16} score={sc:.3f}   {tag}")
print("\nBM25 nails the literal token. Dense (in Colab) is weaker on a bare code with no context.")
print("Lexical precision vs semantic recall -- we want BOTH.")

## §6 — Hybrid retrieval with Reciprocal Rank Fusion (RRF)

How do you combine a BM25 score (unbounded, ~0–15) with a cosine score (0–1)? You **don't add the
scores** — the scales are incompatible and one would drown the other. Instead you fuse the **ranks**.

**Reciprocal Rank Fusion** — for each retriever, a document at rank *r* (1 = best) contributes
`1 / (k + r)`; sum across retrievers:

$$\text{RRF}(d) = \sum_{i} \frac{1}{k + \text{rank}_i(d)} \qquad (k \approx 60,\ \text{a standard damping constant})$$

RRF is beautiful because it is **scale-free and parameter-light**: it only needs each retriever's
*ordering*. A document ranked highly by *either* retriever floats up; a document ranked highly by
*both* wins. That is precisely "dense recall OR lexical precision, and best when both agree".

In [ ]:
def rrf_fuse(rankings, k=60, topn=5):
    # rankings: list of ranked id-lists, best first. Returns fused [(id, rrf_score)].
    fused = {}
    for ranked_ids in rankings:
        for rank, cid in enumerate(ranked_ids, start=1):
            fused[cid] = fused.get(cid, 0.0) + 1.0 / (k + rank)
    return sorted(fused.items(), key=lambda kv: -kv[1])[:topn]

class HybridRetriever:
    name = "hybrid"
    def __init__(self, retrievers, k=60, pool=10):
        self.retrievers = retrievers; self.k = k; self.pool = pool
        self.by_id = {c["id"]: c for c in retrievers[0].chunks}
    def search(self, q, k=5):
        rankings = [[c["id"] for c, _ in r.search(q, self.pool)] for r in self.retrievers]
        fused = rrf_fuse(rankings, k=self.k, topn=k)
        return [(self.by_id[cid], sc) for cid, sc in fused]

hybrid = HybridRetriever([bm25, dense])

for q, gold in [(gap_q, "credentials"), (exact_q, "ratelimits")]:
    top, sc = hybrid.search(q, 1)[0]
    tag = "HIT" if top["doc"] == gold else "MISS"
    print(f"hybrid  {q[:38]:38} -> {top['id']:16} {tag}")
print("\nOne retriever handles the paraphrase (dense), the other the literal token (bm25);")
print("RRF lets the winner of each carry the fused ranking. Robust across BOTH query types.")

## §7 — The scoreboard: measure all four on both eval sets

Two metrics, both standard for retrieval:

- **hit@1** — is the top result from the gold document? (Did we put the right thing first?)
- **MRR** (Mean Reciprocal Rank) — `1/rank` of the first gold-doc hit, averaged. Rewards ranking the
  right doc high even when it isn't #1 (this is what the reranker in L86 then sharpens).

Read the table by *column*: lexical retrievers do fine on the word-overlap `EVAL` and **collapse** on
`PARAPHRASE`; dense holds up on paraphrases; **hybrid is the most robust across both** — which is the
whole point.

In [ ]:
def gold_rank(results, gold):
    for i, (c, _) in enumerate(results, start=1):
        if c["doc"] == gold:
            return i
    return None

def evaluate(retriever, evalset, k=5):
    hit1 = 0.0; mrr = 0.0
    for e in evalset:
        res = retriever.search(e["q"], k=k)
        if res and res[0][0]["doc"] == e["gold"]:
            hit1 += 1
        r = gold_rank(res, e["gold"])
        mrr += (1.0 / r) if r else 0.0
    n = len(evalset)
    return hit1 / n, mrr / n

RETRIEVERS = [tfidf, bm25, dense, hybrid]
print(f"{'retriever':10}{'EVAL hit@1':>12}{'EVAL MRR':>10}{'PARA hit@1':>12}{'PARA MRR':>10}")
print("-" * 54)
SCORE = {}
for r in RETRIEVERS:
    eh, em = evaluate(r, EVAL)
    ph, pm = evaluate(r, PARAPHRASE)
    SCORE[r.name] = {"eval_hit1": eh, "eval_mrr": em, "para_hit1": ph, "para_mrr": pm}
    print(f"{r.name:10}{eh:>12.2f}{em:>10.2f}{ph:>12.2f}{pm:>10.2f}")

if MODEL_LOADED:
    print("\nReal-model reading: lexical (tfidf/bm25) crater on PARA; dense recovers it; hybrid is")
    print("strong on BOTH columns -> the most robust single choice for production.")
else:
    print("\n(Fallback backend: the PARA column will NOT show the real dense win. Run in Colab.)")

# EXPERIMENT: add your own paraphrase to PARAPHRASE above and re-run. Where does each retriever land?

### The RRF damping constant `k`

`k` controls how sharply RRF favours top ranks. Small `k` → the #1 result dominates; large `k` →
lower ranks get more say (good when your retrievers are noisy and you want to pool evidence). 60 is
the classic default from the original RRF paper. Sweep it and watch the fused quality move.

In [ ]:
print(f"{'k':>5}{'PARA hit@1':>12}{'PARA MRR':>10}")
print("-" * 27)
for kk in (1, 10, 60, 200):
    h = HybridRetriever([bm25, dense], k=kk)
    ph, pm = evaluate(h, PARAPHRASE)
    print(f"{kk:>5}{ph:>12.2f}{pm:>10.2f}")
print("\nThere is no universally best k -- you TUNE it on your own eval set (L88 formalizes this).")
# EXPERIMENT: try fusing THREE retrievers -- HybridRetriever([tfidf, bm25, dense]). Better or worse?

## §8 — Practical guidance (what to actually ship)

**Default to hybrid.** In real systems dense+BM25+RRF is the strong, boring default. Pure dense loses
exact-token queries; pure lexical loses paraphrases. Hybrid covers both with almost no downside beyond
running two indexes.

**Picking an embedding model.** `all-MiniLM-L6-v2` is the great starter: tiny, fast, CPU-friendly.
Scale up (e.g. `bge`, `gte`, `e5`, or a hosted embedding API) when your eval says you need more recall.
Always check the **MTEB** leaderboard for retrieval, not general benchmarks — and re-run *your own*
eval (L88), because leaderboard rank rarely survives contact with your domain.

**Normalize, then dot.** L2-normalize embeddings once so cosine is a dot product; it's faster and
your ANN index (below) expects it.

**At scale, swap the numpy matrix for an ANN index.** We used a brute-force `E @ q` here for clarity.
Past ~10^5 chunks you switch to an approximate-nearest-neighbour index (**FAISS**, hnswlib, or a
vector DB like Qdrant/pgvector). The retriever *interface* — `search(q, k)` — stays identical, which
is why we hid everything behind that method.

**Chunking still rules everything (L84).** A retriever can only rank chunks that exist. Dense doesn't
rescue a fact shattered across a boundary. Chunk well first, then retrieve well.

**The mental model:** *dense = recall on meaning; lexical = precision on tokens; RRF = union of their
strengths.* Retrieval casts the net (recall); L86's reranker will tighten precision at the top.

## §9 — Ten retrieval pitfalls

1. **Believing a better lexical algorithm fixes meaning.** BM25 > TF-IDF, but both are still blind to
   paraphrases. Only embeddings close FM2.
2. **Adding raw scores across retrievers.** Cosine (0–1) and BM25 (0–15) live on different scales;
   summing them lets one drown the other. Fuse *ranks* (RRF) or min-max normalize first.
3. **Trusting pure dense for exact tokens.** Error codes, IDs, SKUs, version strings — keep a lexical
   channel or you will miss `HTTP 429`.
4. **Embedding query and document differently.** Some models want an instruction prefix (e.g. e5's
   `"query: "` / `"passage: "`). Mismatched prefixes silently wreck recall.
5. **Forgetting to normalize.** Un-normalized vectors make cosine ≠ dot and corrupt ANN results.
6. **Re-embedding the corpus at query time.** Embed documents once at index time; embed only the query
   at search time. Otherwise latency and cost explode.
7. **Evaluating on lexical-friendly questions only.** If your eval reuses the docs' words, lexical
   looks great and you never discover FM2. Include paraphrases (like `PARAPHRASE`).
8. **Ignoring the embedding model's context limit.** Feed a chunk longer than the model's max tokens
   and the tail is silently truncated — another reason L84's chunk size matters.
9. **Chasing MTEB rank instead of your eval.** Leaderboard order rarely matches your domain; measure
   on your own data (L88).
10. **Skipping the reranker.** Hybrid gets the right chunk into the top-k; it often isn't #1. That last
    reorder is L86's job — don't expect retrieval alone to be perfect.

## §10 — Verification

Deterministic checks run everywhere (they don't depend on the neural model): lexical blindness, RRF
math, and the fusion interface. The semantic-win checks run only when the **real** model is loaded
(in Colab); otherwise they're reported as skipped so the notebook stays green offline.

In [ ]:
checks = []
def check(name, cond):
    checks.append(bool(cond)); print(("PASS " if cond else "FAIL ") + name)

# --- deterministic: lexical retrievers are blind to the paraphrase ---
check("tfidf misses the 'reset password' paraphrase",
      tfidf.search(gap_q, 1)[0][0]["doc"] != "credentials")
check("bm25 misses the 'reset password' paraphrase",
      bm25.search(gap_q, 1)[0][0]["doc"] != "credentials")

# --- deterministic: bm25 nails the exact token ---
check("bm25 finds the exact token 'HTTP 429'",
      bm25.search("HTTP 429", 1)[0][0]["doc"] == "ratelimits")

# --- deterministic: RRF math is correct ---
# doc 'A' is rank1 in list1 & rank2 in list2 -> 1/61 + 1/62; 'B' is rank2 & rank1 -> same total.
fu = dict(rrf_fuse([["A", "B"], ["B", "A"]], k=60, topn=5))
check("RRF sums reciprocal ranks across lists",
      abs(fu["A"] - (1/61 + 1/62)) < 1e-9 and abs(fu["A"] - fu["B"]) < 1e-9)
# a doc ranked #1 by both beats a doc ranked #1 by only one
fu2 = dict(rrf_fuse([["X", "Y"], ["X", "Z"]], k=60, topn=5))
check("RRF rewards agreement (top by both > top by one)", fu2["X"] > fu2["Y"] and fu2["X"] > fu2["Z"])

# --- deterministic: hybrid never underperforms bm25 on the exact-token query ---
check("hybrid keeps the exact-token hit",
      hybrid.search("HTTP 429", 1)[0][0]["doc"] == "ratelimits")

# --- deterministic: lexical collapses on paraphrases vs the lexical-friendly eval ---
check("lexical bm25 scores lower on PARA than on EVAL (the semantic gap, measured)",
      SCORE["bm25"]["para_hit1"] <= SCORE["bm25"]["eval_hit1"])

# --- semantic (real model only): the whole point of the lesson ---
# Reported but NOT hard-asserted, so a rare embedding quirk can never crash your Colab run.
if MODEL_LOADED:
    print("\n-- semantic-win checks (real model) --")
    sem = [
        ("dense closes the gap: finds credentials for 'reset my password'",
         dense.search(gap_q, 1)[0][0]["doc"] == "credentials"),
        ("dense beats bm25 on paraphrase hit@1",
         SCORE["dense"]["para_hit1"] > SCORE["bm25"]["para_hit1"]),
        ("hybrid is at least as good as bm25 on paraphrase hit@1",
         SCORE["hybrid"]["para_hit1"] >= SCORE["bm25"]["para_hit1"]),
    ]
    for nm, cond in sem:
        print(("PASS " if cond else "NOTE ") + nm)
else:
    print("\nSKIP semantic-win checks (fallback backend; run in Colab with the real model)")

# The hard assert covers only the deterministic checks, so the notebook is green everywhere.
print(f"\n{sum(checks)}/{len(checks)} deterministic checks passed")
assert sum(checks) == len(checks), "some deterministic checks failed"
print("All deterministic verification checks passed.")

## Summary

- The retriever we carried since L83 was **lexical** — it matches words, so it fails the **semantic
  gap (FM2)**: everyday phrasing that shares no vocabulary with the docs scores ~0.
- **BM25** is a stronger *lexical* ranker than TF-IDF but does **not** fix FM2 — a better algorithm
  for the wrong problem.
- **Dense embeddings** rank by meaning and close the gap, but can blur **exact tokens** (codes, IDs).
- **Hybrid retrieval (BM25 + dense fused by RRF)** takes recall-on-meaning from dense and
  precision-on-tokens from lexical. RRF fuses *ranks*, so it needs no score calibration.
- Measured on both a lexical-friendly and a paraphrase eval, **hybrid is the most robust** — the
  strong default for production RAG.
- Retrieval casts a wide, accurate net; getting the single best chunk to **rank #1** is the
  reranker's job — **L86**.

### Homework
1. **Add 5 harder paraphrases** to `PARAPHRASE` (idioms, typos, multi-intent). Which retriever degrades
   first? Does hybrid stay robust?
2. **Min-max fusion instead of RRF.** Normalize each retriever's scores to [0,1], then combine as
   `alpha*dense + (1-alpha)*bm25`. Sweep `alpha`. When does score-fusion beat rank-fusion, and when
   does the scale mismatch bite?
3. **Swap the embedding model** (e.g. `BAAI/bge-small-en-v1.5`). Re-run the scoreboard. Did MTEB rank
   predict your result on *this* corpus?
4. **Add exact-token stress queries** ("`TLS 1.3`", "`SOC 2`", "`SAML 2.0`") to a new eval set. Confirm
   the dense-weakness / bm25-strength split, then that hybrid recovers both.
5. **Wire it back into your project.** Replace the L83 `VectorIndex` in your RAG pipeline with this
   `HybridRetriever`, keeping the `search(q, k)` interface. Re-run the L83 baseline eval end-to-end.

### Up next — L86: Cross-encoder reranking
Hybrid retrieval reliably pulls the right chunk into the **top-k**, but it is often ranked #5 or #7,
not #1 — and the LLM reads the top few. A **cross-encoder reranker** reads *(query, chunk)* together
(not as separate vectors) and reorders the shortlist with far more precision than any retriever can.
The pattern: retrieve wide and cheap (today), then rerank narrow and precise (next). That attacks
**FM3 — the right chunk is retrieved but ranked too low.**